# 第11章 バッチ処理と推論速度

複数画像の推論方法と、推論速度を測定・比較する方法を確認します。

## CLIによるディレクトリ推論

In [ ]:
!yolo detect predict model=yolo26n.pt source=images/

## ディレクトリを指定したバッチ処理

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')

# ディレクトリを指定（自動的にバッチ分割される）
results = model(source='images/', batch=16)


## 手動によるバッチ処理

In [ ]:
from ultralytics import YOLO
import glob

model = YOLO('yolo26n.pt')

# 画像パスのリストを取得
image_paths = sorted(glob.glob('images/*.jpg'))
print(f"処理対象: {len(image_paths)}枚")

# 手動でバッチ分割
batch_size = 16
total_detections = 0

for i in range(0, len(image_paths), batch_size):
    batch = image_paths[i:i+batch_size]
    results = model(batch)
    
    for result in results:
        detection_count = len(result.boxes)
        total_detections += detection_count

print(f"総検出数: {total_detections}個")


## ストリーム処理

In [ ]:
results = model('images/', stream=True)

for result in results:
    # 1枚処理するごとに、その結果だけがメモリに読み込まれる
    print(f"{len(result.boxes)}個検出")


## 推論速度の確認

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
results = model('image.jpg')

# 速度情報を取得
speed = results[0].speed
print(f"前処理: {speed['preprocess']:.1f}ms")
print(f"推論: {speed['inference']:.1f}ms")
print(f"後処理: {speed['postprocess']:.1f}ms")

# 合計時間とFPSを計算
total_ms = sum(speed.values())
fps = 1000 / total_ms
print(f"\n合計: {total_ms:.1f}ms")
print(f"FPS: {fps:.1f}")


## 複数画像での平均速度計測

In [ ]:
from ultralytics import YOLO
import glob

model = YOLO('yolo26n.pt')
image_paths = glob.glob('images/*.jpg')

total_ms = 0.0
num_images = 0

for image_path in image_paths:
    results = model(image_path)
    total_ms += sum(results[0].speed.values())
    num_images += 1

avg_time = total_ms / num_images
avg_fps = 1000 / avg_time

print(f"画像数: {num_images}枚")
print(f"平均処理時間（1枚あたり）: {avg_time:.1f}ms")
print(f"平均FPS: {avg_fps:.1f}")


## 動画処理のFPS計測

In [ ]:
from ultralytics import YOLO
import time

model = YOLO('yolo26n.pt')

# 動画処理の時間計測
start_time = time.time()
frame_count = 0

for result in model('video.mp4', stream=True):
    frame_count += 1
    # 必要な処理をここに追加

total_time = time.time() - start_time
overall_fps = frame_count / total_time

print(f"総フレーム数: {frame_count}")
print(f"処理時間: {total_time:.1f}秒")
print(f"平均FPS: {overall_fps:.1f}")


## 最初の実行が遅い

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')

# 1回目
results1 = model('image.jpg')
print(f"1回目: {results1[0].speed['inference']:.1f}ms")

# 2回目
results2 = model('image.jpg')
print(f"2回目: {results2[0].speed['inference']:.1f}ms")

# 3回目
results3 = model('image.jpg')
print(f"3回目: {results3[0].speed['inference']:.1f}ms")


## ウォームアップを含めた計測

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')

# ウォームアップ（1回実行）
_ = model('image.jpg')

# 本番の計測
results = model('image.jpg')
print(f"推論時間: {results[0].speed['inference']:.1f}ms")


## モデルサイズごとの速度比較

In [ ]:
from ultralytics import YOLO

models = ['yolo26n.pt', 'yolo26s.pt', 'yolo26m.pt', 'yolo26l.pt', 'yolo26x.pt']
image_path = 'image.jpg'

for model_name in models:
    model = YOLO(model_name)
    
    # ウォームアップ
    _ = model(image_path)
    
    # 速度計測
    results = model(image_path)
    total_time = sum(results[0].speed.values())
    
    print(f"{model_name}: {total_time:.1f}ms")


## 画像サイズごとの速度比較

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26x.pt')
image_path = 'image.jpg'
sizes = [320, 640, 1280]

# ウォームアップ
_ = model(image_path)

for size in sizes:
    # 速度計測
    results = model(image_path, imgsz=size)
    total_time = sum(results[0].speed.values())
    print(f"{size}x{size}: {total_time:.1f}ms")


## CPU/GPUでの速度比較

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
image_path = 'image.jpg'

# ウォームアップ
_ = model(image_path)

# CPU で推論
results_cpu = model(image_path, device='cpu')
cpu_time = sum(results_cpu[0].speed.values())
print(f"CPU: {cpu_time:.1f}ms")

# GPU で推論（CUDA が利用可能な場合）
results_gpu = model(image_path, device='0')
gpu_time = sum(results_gpu[0].speed.values())
print(f"GPU: {gpu_time:.1f}ms")


## GPU利用可能性の確認

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU利用可能: {torch.cuda.get_device_name(0)}")
    print(f"GPU数: {torch.cuda.device_count()}")
else:
    print("GPUは利用できません")


## バッチ処理の例

In [ ]:
from ultralytics import YOLO
import time

model = YOLO('yolo26n.pt')
image_dir = 'data'

# ウォームアップ
_ = model(image_dir, batch=1)

# バッチサイズ1（明示指定）
start = time.time()
_ = model(image_dir, batch=1)
time_batch1 = time.time() - start

# バッチサイズを大きくして一括処理
batch_size = 32  # GPUメモリに応じて調整
start = time.time()
_ = model(image_dir, batch=batch_size)
time_batch_all = time.time() - start

print(f"batch=1: {time_batch1:.2f}秒")
print(f"batch={batch_size}: {time_batch_all:.2f}秒")
